In [138]:
#Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif

In [139]:
df=pd.read_csv("/content/railway.csv")

In [140]:
#data check data type:
def convert_variables(datatype,var,format="mixed"):
  if datatype=="Datetime":
    df[var]=pd.to_datetime(df[var],errors="coerce",format="mixed")
  elif datatype=="numerical":
    df[var]=pd.to_numeric(df[var],errors="coerce")
  elif datatype=="categorical":
    df[var]=str(df[var])
  else:
    print("invalid datatype")
  return df[var], df.dtypes

In [141]:
#data understanding
def classify_variables(data):
  numerical_col=df.select_dtypes(include=np.number).columns
  datetime_col=df.select_dtypes(include="datetime").columns
  categorical_col=df.select_dtypes(exclude=[np.number,"datetime"]).columns
  return numerical_col,categorical_col, datetime_col

def overview():
  print(f" the total number of object variables = {(df.select_dtypes(include='object').dtypes).count()} \n and the total number of numerical variables = {(df.select_dtypes(include='number').dtypes).count()}")
  print(f"shape: {df.shape} \n the duplicates:{df.duplicate.sum()}")
  print(f"the missing values:{df.isnull.sum()} \n ")
  print(f"the statistical summary for categorical {df.decribe(include="object")} \n")
  print(f"the statistical summary for numerical {df.decribe()} \n")






In [142]:
# data cleanning
def handling_missing_values(variable,values):
    if df[variable].dtypes == 'object':
        df[variable].fillna(values,inplace=True)
        return df[variable].unique()
    elif values == 'mean':
         df[variable].fillna(df[variable].mean(),inplace=True)
         print(f'this is the mean of the {variable} : {df[variable].mean()} \n')
         return df[df[variable]]
    elif values == 'median':
        df[variable].fillna(df[variable].median(),inplace=True)
        print(f'this is the mean of the {variable} : {df[variable].median()} \n')
    elif values == 'mode':
        df[variable].fillna(df[variable].mode(),inplace=True)
        print(f'this is the mode of the {variable} : {df[variable].mode()} \n')
        return df[variable]

In [143]:
def correct_inconsistency(col_name,value_Error,correct_value):
    df[col_name] = np.where(
    df[col_name] == value_Error,
    correct_value,
    df[col_name]
                            )
    return df[col_name].unique()

In [144]:
# =========================================================
# PRINT HELPERS
# =========================================================

def print_section(title):
    print("\n" + "="*60)
    print(title)
    print("="*60)

def print_list(title, items):
    print(f"\n{title}")
    if items:
        for i in items:
            print(f"  - {i}")
    else:
        print("  - None")

def print_distribution(y, title):
    print_section(title)
    dist = y.value_counts(normalize=True) * 100
    for k, v in dist.items():
        print(f"  Class {k}: {v:.2f}%")

In [145]:
# -------------------------------------------------------
# 6. ENCODE CATEGORICAL VARIABLES
# -------------------------------------------------------

def encode_categorical(df, method="frequency", target_variable=None):
    """
    Encode categorical variables.

    Small category variables (<10) → One-Hot Encoding.
    Large category variables → Frequency or Target encoding.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.

    method : str
        Encoding method for large categories:
        "frequency" or "target".

    target_variable : str
        Required when method="target".

    Returns
    -------
    pandas.DataFrame
        New DataFrame with encoded categorical variables.
    """

    new_df = df.copy()

    cat_cols = new_df.select_dtypes(include="object").columns

    # exclude target colomn:

    if target_variable is not None:
        cat_cols = [col for col in cat_cols if col != target_variable]


    if len(cat_cols) == 0:
        return new_df

    for col in cat_cols:

        if new_df[col].nunique() < 10:

            dummies = pd.get_dummies(new_df[col], prefix=col)

            new_df = pd.concat([new_df.drop(col, axis=1), dummies], axis=1)

        else:

            if method == "frequency": #--> Used when categories are many

                freq = new_df[col].value_counts() / len(new_df)

                new_df[col] = new_df[col].map(freq)

            elif method == "target":  #--> Used when target variable exists

                if target_variable is None or target_variable not in new_df.columns:
                    raise ValueError("target_variable must be provided")

                target_mean = new_df.groupby(col)[target_variable].mean()

                new_df[col] = new_df[col].map(target_mean)

    return new_df


In [146]:
# =========================================================
# 1. FILTER SELECTION
# =========================================================

from scipy.stats import chi2_contingency

def filter_selection(df, target_variable, alpha=0.05, del_flag=True):
    """
    Filter selection following the PDF logic:
    - Categorical predictors vs categorical target -> Chi-square
    - Numeric / ordinal predictors vs target -> Mutual Information
    """

    print("\n" + "=" * 70)
    print("FILTER SELECTION START")
    print("=" * 70)

    new_df = df.copy()

    if target_variable not in new_df.columns:
        raise ValueError("Target variable not found")

    X = new_df.drop(columns=[target_variable]).copy()
    y = new_df[target_variable].copy()
    removed_features = []

    print("Original dataframe shape:", new_df.shape)
    print("\nTarget variable:", target_variable)
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("y dtype:", y.dtype)

    # -------------------------------------------------
    # Encode target only for MI
    # -------------------------------------------------

    #  checks if the target column is stored as strings or Categorical

    is_classification = (
        pd.api.types.is_object_dtype(y) or pd.api.types.is_categorical_dtype(y)
    )

    if is_classification:
        print("Task type: classification")
        y_encoded = LabelEncoder().fit_transform(y.astype(str).fillna("MISSING"))
    else:
        print("Task type: regression")
        y_encoded = pd.to_numeric(y, errors="coerce").fillna(y.median())

    # -------------------------------------------------
    # 1) Categorical predictors -> Chi-square if target categorical
    #    If target numeric, use MI after one-hot encoding
    # -------------------------------------------------
    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

    print("\n--- STEP 1: Categorical predictors ---")

    for col in categorical_cols:
        s = X[col].astype("object").fillna("MISSING")

        if s.nunique() < 2:
            continue

        if is_classification:

# If the target y is categorical (classification task):
# Build a contingency table (ct) between the predictor s and the target y.

            ct = pd.crosstab(s.astype(str), y.astype(str).fillna("MISSING"))

            if ct.shape[0] < 2 or ct.shape[1] < 2:
                continue

 # Chi-square test of independence
            _, p_val, _, _ = chi2_contingency(ct)

            print(f"{col} -> p-value = {p_val:.6f}")

            if p_val >= alpha:
                removed_features.append(col)
                if del_flag:
                    X = X.drop(columns=[col], errors="ignore")
                print(f"→ REMOVE {col}")
            else:
                print(f"→ KEEP {col}")

        else:
            # Numeric target: use MI on one-hot encoded predictor,
            #One‑hot encode the categorical predictor into dummy variables.
            dummies = pd.get_dummies(s.astype(str), prefix=col, drop_first=False)

            if dummies.shape[1] == 0:
                continue

            mi_vals = mutual_info_regression(dummies.astype(float), y_encoded)
            mi_score = float(np.mean(mi_vals)) # Take the mean MI score across dummy columns.

            print(f"{col} -> mean MI = {mi_score:.6f}")

            if mi_score < 0.005:
                removed_features.append(col)
                if del_flag:
                    X = X.drop(columns=[col], errors="ignore")
                print(f"→ REMOVE {col}")
            else:
                print(f"→ KEEP {col}")

    # -------------------------------------------------
    # 2) Numeric / ordinal predictors -> MI
    # -------------------------------------------------
    print("\n--- STEP 2: Numeric / ordinal predictors ---")

    numeric_cols = X.select_dtypes(include="number").columns.tolist()

    print("Numeric vars (for MI):", numeric_cols)

    if len(numeric_cols) > 0:
        X_num = X[numeric_cols].copy()

        if is_classification:
            mi = mutual_info_classif(X_num, y_encoded, random_state=42)
        else:
            mi = mutual_info_regression(X_num, y_encoded, random_state=42)


        # converts raw MI results into a readable table, then sorts them.

        mi_series = pd.Series(mi, index=numeric_cols)

        # Map MI values to features

        print("\nMI values:")
        print(mi_series.sort_values())


        low_mi = mi_series[mi_series < 0.005].index.tolist()

        if low_mi:
            print("Low MI features:", low_mi)
            removed_features.extend(low_mi)

            if del_flag:
                X = X.drop(columns=low_mi, errors="ignore")

    print("\n--- FINAL RESULT ---")
    print("Removed features:", removed_features)
    print("Remaining shape:", X.shape)

    if X.shape[1] == 0:
        print("No features left")
        return new_df[[target_variable]]

    return pd.concat([X, y], axis=1)


In [147]:
"""
code
"""
# data combination
df["Purchase Datetime"] = df["Date of Purchase"] + " " + df["Time of Purchase"]
df["Journey Datetime"] = df["Date of Journey"] + " " + df["Departure Time"]
column_deleted = df.columns[df.columns.str.endswith("ID")]
columns_to_drop = list(column_deleted) + ['Date of Purchase', 'Time of Purchase', 'Date of Journey', 'Departure Time']
df.drop(columns=columns_to_drop, inplace=True)


convert_variables("Datetime","Purchase Datetime")
convert_variables("Datetime","Journey Datetime")

handling_missing_values('Reason for Delay',"No delay")
handling_missing_values('Railcard',"Haven't Railcard")

/tmp/ipykernel_1578/2298408797.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[variable].fillna(values,inplace=True)


array(['Adult', "Haven't Railcard", 'Disabled', 'Senior'], dtype=object)

In [148]:
# check logical

x= df[df["Departure Station"] == df["Arrival Destination"]]
print("Number of matching journeys:", len(x))
print("the numer of tickets that the price negative:",(df['Price'] < 0).sum())
print("the Number of purchase after journey:",(df["Purchase Datetime"] > df["Journey Datetime"]).sum())



Number of matching journeys: 0
the numer of tickets that the price negative: 0
the Number of purchase after journey: 0


In [149]:

# correct inconsistency
correct_inconsistency("Reason for Delay","Staffing","Staff Shortage")
correct_inconsistency("Reason for Delay","Weather","Weather Conditions")
correct_inconsistency("Reason for Delay","Staffing","Staff Shortage")

array(['No delay', 'Signal Failure', 'Technical Issue',
       'Weather Conditions', 'Staff Shortage', 'Signal failure',
       'Traffic'], dtype=object)

In [154]:
# =========================================================
# FEATURE SELECTION
# =========================================================

print_section("FEATURE SELECTION")
df_clean=df.copy()
target = "Refund Request"
 #Categorical vs Target
# =========================================================
numeric_vars, ordinal_vars, categorical_vars = classify_variables(df_clean)
for cat in categorical_vars:
    if cat != target:
        cross = pd.crosstab(df_clean[cat], df_clean[target], normalize='index') * 100
        print(f"\n{cat} vs {target}:")
        print(cross)
# 1) FILTER
df_fs = filter_selection(df_clean, target)
df_clean=df_fs.copy()



FEATURE SELECTION

Purchase Datetime vs Refund Request:
Refund Request          No  Yes
Purchase Datetime              
2023-12-08 12:41:11  100.0  0.0
2023-12-16 11:23:01  100.0  0.0
2023-12-19 19:51:27  100.0  0.0
2023-12-20 23:00:36  100.0  0.0
2023-12-27 18:22:56  100.0  0.0
...                    ...  ...
2024-04-30 18:42:58  100.0  0.0
2024-04-30 18:46:10  100.0  0.0
2024-04-30 18:56:41  100.0  0.0
2024-04-30 19:51:47  100.0  0.0
2024-04-30 20:05:39  100.0  0.0

[31568 rows x 2 columns]

Journey Datetime vs Refund Request:
Refund Request          No  Yes
Journey Datetime               
2024-01-01 00:00:00  100.0  0.0
2024-01-01 01:30:00  100.0  0.0
2024-01-01 01:45:00  100.0  0.0
2024-01-01 02:15:00  100.0  0.0
2024-01-01 02:45:00  100.0  0.0
...                    ...  ...
2024-04-30 22:15:00  100.0  0.0
2024-04-30 22:30:00  100.0  0.0
2024-04-30 23:00:00  100.0  0.0
2024-04-30 23:30:00  100.0  0.0
2024-04-30 23:45:00  100.0  0.0

[8732 rows x 2 columns]

FILTER SELECTION START

In [153]:
# 5. Export and download
from google.colab import files
cleaned_filename = 'railway_cleaned.xlsx'
df_clean.to_excel(cleaned_filename, index=False, engine='openpyxl')
files.download(cleaned_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [155]:
df_clean

,Purchase Type,Payment Method,Railcard,Ticket Type,Price,Departure Station,Arrival Destination,Arrival Time,Actual Arrival Time,Journey Status,Reason for Delay,Purchase Datetime,Journey Datetime,Refund Request
0,Online,Contactless,Adult,Advance,43,London Paddington,Liverpool Lime Street,13:30:00,13:30:00,On Time,No delay,2023-12-08 12:41:11,2024-01-01 11:00:00,No
1,Station,Credit Card,Adult,Advance,23,London Kings Cross,York,11:35:00,11:40:00,Delayed,Signal Failure,2023-12-16 11:23:01,2024-01-01 09:45:00,No
2,Online,Credit Card,Haven't Railcard,Advance,3,Liverpool Lime Street,Manchester Piccadilly,18:45:00,18:45:00,On Time,No delay,2023-12-19 19:51:27,2024-01-02 18:15:00,No
3,Station,Credit Card,Haven't Railcard,Advance,13,London Paddington,Reading,22:30:00,22:30:00,On Time,No delay,2023-12-20 23:00:36,2024-01-01 21:30:00,No
4,Online,Contactless,Haven't Railcard,Advance,76,Liverpool Lime Street,London Euston,19:00:00,19:00:00,On Time,No delay,2023-12-27 18:22:56,2024-01-01 16:45:00,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31648,Online,Credit Card,Haven't Railcard,Off-Peak,4,Manchester Piccadilly,Liverpool Lime Street,20:30:00,20:30:00,On Time,No delay,2024-04-30 18:42:58,2024-04-30 20:00:00,No
31649,Online,Contactless,Haven't Railcard,Off-Peak,10,London Euston,Birmingham New Street,21:35:00,21:35:00,On Time,No delay,2024-04-30 18:46:10,2024-04-30 20:15:00,No
31650,Station,Credit Card,Haven't Railcard,Off-Peak,4,Manchester Piccadilly,Liverpool Lime Street,20:45:00,20:45:00,On Time,No delay,2024-04-30 18:56:41,2024-04-30 20:15:00,No
31651,Station,Credit Card,Haven't Railcard,Off-Peak,10,London Euston,Birmingham New Street,22:35:00,22:35:00,On Time,No delay,2024-04-30 19:51:47,2024-04-30 21:15:00,No
